# 03 · GSD / SNR / MTF 退化扫描（阶段 B · H1/H2/H3）

**目标**：在 M1 模型上做 GSD（1.0–8.0×）、SNR（30→6 dB）、MTF（σ 0.5–4.0）三组扫描，
与 02 云扫描合起来凑齐 M2 四条响应曲线，验证 H1/H2/H3。

**前置**：
1. 基于 M1（01_terratorch_eurosat_baseline.ipynb）训练出的模型。
2. 通过右侧 + Add Input 添加 **整个 eo-degrade 仓库**（Dataset 类型，含 `degrade/` 与 `scripts/`）。
3. EuroSAT 数据（M1 已下载到 data/ 或 Add Input 添加）。

**可复现性**：全部扫描固定 `SEED=0`；SNR 扫描点由数据均值反解 (a,b) 生成（横轴 dB 均匀）。

In [ ]:
# 环境准备
!pip install -q terratorch torchgeo 2>&1 | tail -1
import torch
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 挂载 eo-degrade 仓库（degrade 库 + scripts 扫描引擎）
import sys, os, glob

candidates = []
for root in ['/kaggle/input', '.']:
    for hit in glob.glob(os.path.join(root, '**', 'scripts', 'scan_degradation.py'), recursive=True):
        candidates.append(os.path.dirname(os.path.dirname(hit)))

if candidates:
    repo = candidates[0]
    sys.path.insert(0, repo)
    print('找到 eo-degrade 仓库:', repo)
else:
    raise FileNotFoundError('未找到 scripts/scan_degradation.py。请添加整个 eo-degrade 仓库（Dataset 类型）。')

from degrade.gsd import degrade_gsd
from degrade.snr import add_poisson_gaussian
from degrade.mtf import degrade_mtf
from scripts.scan_degradation import (
    gsd_scan_points, snr_scan_points_for, mtf_scan_points, mtf_kernel_size,
    eval_degrade, run_scan,
)
print('退化库与扫描引擎加载成功')

In [ ]:
# 加载 M1 模型（与 02 相同逻辑）
from terratorch.tasks import ClassificationTask

CKPT_PATH = None
if CKPT_PATH is None:
    hits = glob.glob('/kaggle/**/*.ckpt', recursive=True)
    if hits:
        CKPT_PATH = hits[0]
        print('自动找到 checkpoint:', CKPT_PATH)

MODEL_ARGS = dict(
    model_args={
        'backbone': 'prithvi_eo_v1_100',
        'backbone_kwargs': {'pretrained': True, 'num_frames': 1, 'bands': ['BLUE','GREEN','RED']},
        'decoder': 'IdentityDecoder',
        'num_classes': 10,
    },
    model_factory='EncoderDecoderFactory',
    loss='ce',
    lr=1e-4,
)

if CKPT_PATH is not None:
    try:
        model = ClassificationTask.load_from_checkpoint(CKPT_PATH, map_location='cpu')
        print('从 checkpoint 加载成功（精度应为 M1 水平 ~89%）')
    except Exception as e:
        print('checkpoint 加载失败，回退 pretrained 模式:', e)
        model = ClassificationTask(**MODEL_ARGS)
else:
    print('未找到 checkpoint，使用 pretrained 模型（精度较低，仅验证流程）')
    model = ClassificationTask(**MODEL_ARGS)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
model.eval()
print('模型就绪:', device)

In [ ]:
# 加载 EuroSAT 测试集 + 统计数据均值（SNR 反解需要）
from terratorch.datamodules import EuroSATDataModule

dm = EuroSATDataModule(
    root='data',
    batch_size=32,
    num_workers=2,
    bands=['BLUE', 'GREEN', 'RED'],
    test_transform=[{'class_path': 'albumentations.Resize', 'init_args': {'height': 224, 'width': 224}}],
)
dm.setup('test')
loader = dm.test_dataloader()
print('测试集 batch 数:', len(loader))

# 值域诊断 + 全测试集均值（SNR 反解用）
xb = next(iter(loader))['image']
print('image 范围: min=%.3f max=%.3f' % (xb.min().item(), xb.max().item()))
GLOBAL_MAX = xb.max().item()

sum_im, n_pix = 0.0, 0
with torch.no_grad():
    for bch in loader:
        im = bch['image']
        if GLOBAL_MAX > 1.5:
            im = im / 255.0
        sum_im += im.sum().item()
        n_pix += im.numel()
X_MEAN = sum_im / n_pix
print('数据均值 X_MEAN = %.4f' % X_MEAN)

In [ ]:
# GSD 扫描
SEED = 0
print('=== GSD 扫描（1.0×–8.0×）===')
gsd_results = []
for s in gsd_scan_points():
    acc, f1 = eval_degrade(model, loader, lambda im, s=s: degrade_gsd(im, s))
    gsd_results.append({'degrade': s, 'accuracy': round(float(acc), 4), 'f1': round(float(f1), 4)})
    print(f'GSD {s:.1f}× → acc {acc:.4f} f1 {f1:.4f}')

In [ ]:
# SNR 扫描（目标 30→6 dB，a 由数据均值反解）
snr_points = snr_scan_points_for(X_MEAN)
print('=== SNR 扫描（目标 dB → a 反解）===')
snr_results = []
for pt in snr_points:
    acc, f1 = eval_degrade(
        model, loader,
        lambda im, pt=pt: add_poisson_gaussian(im, a=pt['a'], b=pt['b'], seed=SEED),
    )
    snr_results.append({'degrade': pt['db'], 'accuracy': round(float(acc), 4), 'f1': round(float(f1), 4)})
    print(f"SNR {pt['db']:.0f}dB (a={pt['a']:.5f}) → acc {acc:.4f} f1 {f1:.4f}")

In [ ]:
# MTF 扫描（σ 0.5–4.0，核尺寸自适应）
print('=== MTF 扫描（σ 0.5–4.0）===')
mtf_results = []
for s in mtf_scan_points():
    ks = mtf_kernel_size(s)
    acc, f1 = eval_degrade(model, loader, lambda im, s=s, ks=ks: degrade_mtf(im, s, kernel_size=ks))
    mtf_results.append({'degrade': s, 'accuracy': round(float(acc), 4), 'f1': round(float(f1), 4)})
    print(f'MTF σ={s:.1f} (kernel {ks}) → acc {acc:.4f} f1 {f1:.4f}')

In [ ]:
# 保存三组 CSV（Kaggle 输出目录）
import csv
os.makedirs('/kaggle/working/results', exist_ok=True)

def save_csv(path, rows):
    with open(path, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=['degrade', 'accuracy', 'f1'])
        w.writeheader()
        w.writerows(rows)
    print('已保存:', path)

save_csv('/kaggle/working/results/gsd_scan.csv', gsd_results)
save_csv('/kaggle/working/results/snr_scan.csv', snr_results)
save_csv('/kaggle/working/results/mtf_scan.csv', mtf_results)

In [ ]:
# 汇总曲线图（三条曲线；若 02 云扫描 CSV 已放入本 notebook 输入则一并绘制）
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

def plot(ax, rows, xlab, ylab, title):
    xs = [r['degrade'] for r in rows]
    ys = [r['accuracy'] * 100 for r in rows]
    ax.plot(xs, ys, 'o-', color='#2B5FB8', linewidth=2)
    ax.set_xlabel(xlab); ax.set_ylabel(ylab); ax.set_title(title)
    ax.grid(alpha=0.3)

plot(axes[0], gsd_results, 'GSD 退化倍数 (×)', '精度 (%)', 'H1: GSD 响应')
plot(axes[1], snr_results, 'SNR (dB)', '精度 (%)', 'H2: SNR 响应')
plot(axes[2], mtf_results, 'MTF σ', '精度 (%)', 'H3: MTF 响应')
plt.tight_layout()
png_path = '/kaggle/working/results/gsd_snr_mtf_curves.png'
plt.savefig(png_path, dpi=150, bbox_inches='tight')
plt.show()
print('PNG 已保存:', png_path)

## 结果回填

跑完后把三组 CSV 和曲线图下载回仓库：
- `results/gsd_scan.csv`、`results/snr_scan.csv`、`results/mtf_scan.csv`
- `results/gsd_snr_mtf_curves.png`

然后在 README 更新 H1/H2/H3 结论（甜区/饱和点/高频伤害，或阴性结果）。

**口径说明**：SNR 为功率比 10·log10(信号功率/噪声方差)，在归一化影像上由数据均值反解 (a,b) 得到；
低 dB 时实测 SNR 可能略高于目标（泊松噪声非纯高斯），如需精确可后续用实测值重绘横轴。